<a href="https://colab.research.google.com/github/Akutabox/rtm1_2501010140_Billy/blob/main/notebooks/Akusisi_OpenMateo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import json
import time
import hashlib
import os
import csv
import zipfile

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

print("Persiapan berhasil!")

Persiapan berhasil!


In [2]:
NAMA = "Billy Maulana Hasan"
NIM = "2501010140"
EMAIL = "InstikiBali@gmail.com"

PROJECT = "rtm1_" + NIM

BASE = Path("/content") / PROJECT

NOTEBOOK_DIR = BASE / "notebooks"
RAW_DIR = BASE / "data" / "raw"
DOCS_DIR = BASE / "docs"

for folder in [NOTEBOOK_DIR, RAW_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folder tugas berhasil dibuat!")
print(BASE)

Folder tugas berhasil dibuat!
/content/rtm1_2501010140


In [3]:
lokasi = [
    {
        "nama": "Pantai Kuta",
        "latitude": -8.7184,
        "longitude": 115.1686
    },
    {
        "nama": "Tanah Lot",
        "latitude": -8.6212,
        "longitude": 115.0868
    },
    {
        "nama": "Ubud",
        "latitude": -8.5069,
        "longitude": 115.2625
    },
    {
        "nama": "GWK",
        "latitude": -8.8105,
        "longitude": 115.1668
    },
    {
        "nama": "Pantai Sanur",
        "latitude": -8.6883,
        "longitude": 115.2625
    },
    {
        "nama": "Kintamani",
        "latitude": -8.2480,
        "longitude": 115.3750
    }
]

latitude = ",".join(str(x["latitude"]) for x in lokasi)
longitude = ",".join(str(x["longitude"]) for x in lokasi)

print("Jumlah destinasi:", len(lokasi))

Jumlah destinasi: 6


In [4]:
URL = "https://api.open-meteo.com/v1/forecast"

PARAMS = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": "temperature_2m,precipitation,cloud_cover,wind_speed_10m",
    "timezone": "Asia/Makassar",
    "forecast_days": 7,
    "wind_speed_unit": "kmh"
}

HEADERS = {
    "User-Agent": f"AkuisisiData-Tugas1/{NIM} ({EMAIL})"
}

response = None


time.sleep(1)

for percobaan in range(4):
    try:
        response = requests.get(
            URL,
            params=PARAMS,
            headers=HEADERS,
            timeout=15
        )

        if response.status_code in [429, 500, 502, 503, 504]:
            response.raise_for_status()

        response.raise_for_status()
        break

    except requests.RequestException as error:
        if percobaan == 3:
            raise error

        waktu_tunggu = 2 ** percobaan

        print("Request gagal:", error)
        print("Mencoba kembali dalam", waktu_tunggu, "detik")

        time.sleep(waktu_tunggu)

print("Status Code:", response.status_code)
print("Pengambilan data berhasil!")

Status Code: 200
Pengambilan data berhasil!


In [5]:
data_json = response.json()

waktu_akses = datetime.now(
    ZoneInfo("Asia/Makassar")
)

tanggal_file = waktu_akses.strftime("%Y%m%d")

nama_json = f"openmeteo_bali_{tanggal_file}.json"

path_json = RAW_DIR / nama_json

with open(path_json, "wb") as file:
    file.write(response.content)

print("JSON mentah berhasil disimpan:", nama_json)

JSON mentah berhasil disimpan: openmeteo_bali_20261116.json


In [6]:
if isinstance(data_json, dict):
    hasil_lokasi = [data_json]
else:
    hasil_lokasi = data_json

if len(hasil_lokasi) != len(lokasi):
    raise ValueError("Jumlah lokasi dari API tidak sesuai!")

semua_data = []

for i, hasil in enumerate(hasil_lokasi):

    hourly = hasil["hourly"]

    for j, waktu in enumerate(hourly["time"]):

        semua_data.append({
            "lokasi": lokasi[i]["nama"],
            "waktu_wita": waktu,
            "suhu_c": hourly["temperature_2m"][j],
            "curah_hujan_mm": hourly["precipitation"][j],
            "tutupan_awan_persen": hourly["cloud_cover"][j],
            "kecepatan_angin_kmh": hourly["wind_speed_10m"][j]
        })

df = pd.DataFrame(semua_data)

nama_csv = f"openmeteo_bali_{tanggal_file}.csv"

path_csv = RAW_DIR / nama_csv

df.to_csv(path_csv, index=False, encoding="utf-8-sig")

print("Jumlah record:", len(df))
print("Jumlah kolom:", len(df.columns))

display(df.head())

Jumlah record: 1008
Jumlah kolom: 6


,lokasi,waktu_wita,suhu_c,curah_hujan_mm,tutupan_awan_persen,kecepatan_angin_kmh
0,Pantai Kuta,2026-11-16T00:00,26.7,0.0,97,3.0
1,Pantai Kuta,2026-11-16T01:00,26.7,0.0,100,3.2
2,Pantai Kuta,2026-11-16T02:00,26.5,0.1,100,5.1
3,Pantai Kuta,2026-11-16T03:00,25.8,0.0,100,3.3
4,Pantai Kuta,2026-11-16T04:00,25.5,0.0,100,3.2


In [7]:
assert len(df) >= 100, "Record belum mencapai 100!"
assert len(df.columns) >= 5, "Kolom belum mencapai 5!"
assert df["lokasi"].nunique() == 6, "Lokasi belum lengkap!"

print("VALIDASI BERHASIL")
print("Jumlah record:", len(df))
print("Jumlah kolom:", len(df.columns))
print("Jumlah lokasi:", df["lokasi"].nunique())

print("\nJumlah data per lokasi:")
print(df.groupby("lokasi").size())

VALIDASI BERHASIL
Jumlah record: 1008
Jumlah kolom: 6
Jumlah lokasi: 6

Jumlah data per lokasi:
lokasi
GWK             168
Kintamani       168
Pantai Kuta     168
Pantai Sanur    168
Tanah Lot       168
Ubud            168
dtype: int64


In [8]:
def informasi_file(path):
    isi = path.read_bytes()

    return {
        "nama_berkas": path.name,
        "ukuran_byte": len(isi),
        "hash_md5": hashlib.md5(isi).hexdigest()
    }

provenance = {
    "nama_pengambil": NAMA,
    "nim": NIM,
    "sumber": "Open-Meteo Forecast API",
    "url": URL,
    "lisensi": "CC BY 4.0",
    "atribusi": "Weather data by Open-Meteo.com",
    "waktu_akuisisi": waktu_akses.isoformat(),
    "zona_waktu": "Asia/Makassar",
    "jumlah_lokasi": len(lokasi),
    "jumlah_record": len(df),
    "parameter": PARAMS,
    "berkas": [
        informasi_file(path_json),
        informasi_file(path_csv)
    ]
}

with open(DOCS_DIR / "provenance.json", "w", encoding="utf-8") as file:
    json.dump(provenance, file, indent=4, ensure_ascii=False)

print("provenance.json berhasil dibuat!")

provenance.json berhasil dibuat!


In [22]:
checklist = [
    ["1", "Sumber data publik dan legal", "Ya",
     "Open-Meteo: https://open-meteo.com/"],

    ["2", "Data relevan dengan sektor pariwisata", "Ya",
     "Enam destinasi wisata Bali"],

    ["3", "Lisensi sumber diperiksa", "Ya",
     "https://open-meteo.com/en/terms"],

    ["4", "Atribusi sumber dicantumkan", "Ya",
     "Weather data by Open-Meteo.com"],

    ["5", "Tidak mengambil data pribadi", "Ya",
     "Data hanya berupa informasi cuaca"],

    ["6", "Hanya mengambil kolom yang diperlukan", "Ya",
     "Suhu, hujan, awan, dan kecepatan angin"],

    ["7", "Menggunakan User-Agent yang jelas", "Ya",
     "Nama dan email pengambil"],

    ["8", "Menggunakan timeout dan penanganan error", "Ya",
     "Timeout 15 detik dan retry"],

    ["9", "Menghormati batas penggunaan API", "Ya",
     "Dokumentasi: https://open-meteo.com/en/terms"]
]

with open(
    DOCS_DIR / "data_ethics_checklist.csv",
    "w",
    newline="",
    encoding="utf-8-sig"
) as file:

    writer = csv.writer(file)
    writer.writerow(["No", "Pemeriksaan Etika", "Status", "Bukti"])
    writer.writerows(checklist)

print("Data Ethics Checklist berhasil diperbarui!")

Data Ethics Checklist berhasil diperbarui!


In [23]:
readme = f"""
# Tugas 1 - DAM

## Cuaca per Jam di Enam Destinasi Wisata Bali

### 1. Sumber Data

- Nama: Open-Meteo Forecast API
- URL: https://api.open-meteo.com/v1/forecast
- Penerbit: Open-Meteo
- Lisensi: CC BY 4.0
- Atribusi: Weather data by Open-Meteo.com

### 2. Tanggal dan Jam Akses

- Waktu akses: {waktu_akses.strftime("%Y-%m-%d %H:%M:%S")}
- Zona waktu: WITA (Asia/Makassar)

### 3. Metode Akuisisi

Menggunakan API resmi Open-Meteo dengan Python Requests.

Endpoint:
https://api.open-meteo.com/v1/forecast

Parameter:
- latitude dan longitude: koordinat enam destinasi
- hourly: temperature_2m, precipitation, cloud_cover, wind_speed_10m
- timezone: Asia/Makassar
- forecast_days: 7
- wind_speed_unit: kmh

Pengambilan data menggunakan timeout 15 detik,
User-Agent, pemeriksaan status code, serta retry
dengan exponential backoff apabila terjadi gangguan.

### 4. Batasan Etika

- Sumber data menggunakan Open-Meteo API.
- Lisensi data CC BY 4.0.
- Atribusi sumber dicantumkan.
- Tidak mengambil data pribadi.
- Pengambilan dilakukan dalam satu request batch.
- Penggunaan API mengikuti prinsip fair use.
- Tidak menggunakan API key karena tidak diperlukan
  untuk penggunaan non-komersial pada endpoint ini.

### 5. Penjelasan Kolom Dataset

| Kolom | Keterangan |
|---|---|
| lokasi | Nama destinasi wisata |
| waktu_wita | Waktu perkiraan cuaca |
| suhu_c | Suhu udara dalam Celsius |
| curah_hujan_mm | Curah hujan dalam milimeter |
| tutupan_awan_persen | Persentase tutupan awan |
| kecepatan_angin_kmh | Kecepatan angin km/jam |

Jumlah lokasi: 6
Jumlah record: {len(df)}
Jumlah kolom: {len(df.columns)}

Data mencakup perkiraan cuaca per jam selama tujuh hari.
Data merupakan hasil perkiraan model cuaca, bukan
pengukuran langsung dari stasiun cuaca.

### 6. Cara Menjalankan Ulang

1. Instal library:
   pip install pandas requests

2. Buka notebooks/Akusisi_OpenMateo.ipynb

3. Isi nama, NIM, dan email kampus.

4. Jalankan seluruh cell dari awal.

5. Hasil akuisisi tersimpan pada data/raw/
   dan dokumentasi tersimpan pada docs/.

## Referensi

Open-Meteo: https://open-meteo.com/
Lisensi: https://creativecommons.org/licenses/by/4.0/

Weather data by Open-Meteo.com
"""

(BASE / "README.md").write_text(readme, encoding="utf-8")

print("README.md berhasil dibuat!")

README.md berhasil dibuat!


In [11]:
for root, dirs, files in os.walk(BASE):
    level = root.replace(str(BASE), "").count(os.sep)

    print("  " * level + os.path.basename(root) + "/")

    for file in files:
        print("  " * (level + 1) + file)

rtm1_2501010140/
  README.md
  data/
    raw/
      openmeteo_bali_20261116.csv
      openmeteo_bali_20261116.json
  notebooks/
  docs/
    provenance.json
    data_ethics_checklist.csv


In [12]:
import os

print("FILE DATA:")
print(os.listdir(RAW_DIR))

print("\nFILE DOKUMENTASI:")
print(os.listdir(DOCS_DIR))

print("\nJUMLAH DATA:")
print(len(df), "record")

FILE DATA:
['openmeteo_bali_20261116.csv', 'openmeteo_bali_20261116.json']

FILE DOKUMENTASI:
['provenance.json', 'data_ethics_checklist.csv']

JUMLAH DATA:
1008 record


In [20]:
from google.colab import userdata
import subprocess
import shutil
from pathlib import Path

TOKEN = userdata.get("GITHUB_TOKEN")

# Memperbaiki karakter tersembunyi zero-width space pada kata "Akutabox"
REPO = "Akutabox/rtm1_2501010140_Billy"
REPO_URL = f"https://{TOKEN}@github.com/{REPO}.git"

LOCAL_REPO = Path("/content/github_upload")

# Hapus folder lama jika proses clone sebelumnya gagal setengah jalan
if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)

# Clone repository
subprocess.run(
    ["git", "clone", REPO_URL, str(LOCAL_REPO)],
    check=True,
    capture_output=True,
    text=True
)

# Salin data dan dokumentasi
shutil.copytree(BASE / "data", LOCAL_REPO / "data", dirs_exist_ok=True)
shutil.copytree(BASE / "docs", LOCAL_REPO / "docs", dirs_exist_ok=True)

# Salin README
shutil.copy2(BASE / "README.md", LOCAL_REPO / "README.md")

# Atur identitas commit
subprocess.run(["git", "-C", str(LOCAL_REPO), "config",
                "user.name", NAMA], check=True)

subprocess.run(["git", "-C", str(LOCAL_REPO), "config",
                "user.email", EMAIL], check=True)

# Upload
subprocess.run(
    ["git", "-C", str(LOCAL_REPO), "add", "data", "docs", "README.md"],
    check=True
)

subprocess.run(
    ["git", "-C", str(LOCAL_REPO), "commit", "-m",
     "Menambahkan hasil akuisisi dan dokumentasi"],
    check=True
)

subprocess.run(
    ["git", "-C", str(LOCAL_REPO), "push", "origin", "main"],
    check=True
)

# Hapus token dari konfigurasi remote lokal
subprocess.run(
    ["git", "-C", str(LOCAL_REPO), "remote", "set-url",
     "origin", f"https://github.com/{REPO}.git"],
    check=True
)

print("Berhasil mengirim data dan dokumentasi ke GitHub!")

Berhasil mengirim data dan dokumentasi ke GitHub!


In [21]:
print("Jumlah record:", len(df))
print("Jumlah kolom:", len(df.columns))
print("Jumlah lokasi:", df["lokasi"].nunique())

Jumlah record: 1008
Jumlah kolom: 6
Jumlah lokasi: 6
